In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import rasterio
#import pygrib   # this package only runs on linux
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from collections import OrderedDict
from shapely.geometry import Polygon
import matplotlib.cm as cm
from matplotlib.ticker import MultipleLocator
import cartopy.io.img_tiles as cimgt
import geopandas
from scipy.stats import norm as snorm
from matplotlib.colors import Normalize
import time
from shapely.geometry import box

In [ ]:
# test, no need to run
grbs = pygrib.open('gribfiles/jan2023.grib')
print("grbs file opened")
#grbs.seek(0) # grb1 = grbs[1]

In [ ]:
# test, no need to run
jan20 = pygrib.open('gribfiles/USA2013-2023/2020jan.grib')
dec20 = pygrib.open('gribfiles/USA2013-2023/2020dec.grib')

In [ ]:
# This is for getting min temp in an area (all timepoints). if only 1 loc, then get the loc's data
# URB loc:  lat_min = 40.09, lat_max = 40.15, lon_min = -88.25, lon_max= -88.15
# DIX spring: 37.6, -88.7, lat_min = 37.55, lat_max = 37.65, lon_min = -88.75, lon_max= -88.65
# spooner WI = 45.82, -91.87 , lat_min = 45.75, lat_max = 45.85, lon_min = -91.95, lon_max= -91.85
def get_annual_all(grbs_filename_list, output_file, lat_min = 37.55, lat_max = 37.65, lon_min = -88.75, lon_max= -88.65, convert_long3 = False):
    grbs_list = []
    for grbs_filename in grbs_filename_list:
        if os.path.exists(grbs_filename):
            print('opening file' + grbs_filename + '...')
            grbs = pygrib.open(grbs_filename); grbs_list.append(grbs)
            print(grbs_filename + 'is opened')
        else:
            print('File not found: ' + grbs_filename + '. Skipping...')
    print('opened all grbs file required for the minimum calculation, number of files = ' + str(len(grbs_list)))
    print('filtering for locations, retrieving location indices...')
    location_indices, loc_lats, loc_lons = find_location_indices(lat_min, lat_max, lon_min, lon_max, grbs_list[0], convert_long3)
    print('location indices are done, pulling temperature at all time points...')
    loc_all_values, timepoints = retrieve_values_multi_grbs(grbs_list, location_indices)
    print('All time temperatures are found')
    print(loc_all_values)
    all_loc_mins_kelvin = np.min(loc_all_values, axis = 1) # since NAs were filled by 9999, doesn't affect the min
    all_loc_mins_celsius = all_loc_mins_kelvin - 273.15
    loc_temp_table  = pd.DataFrame({'temp': all_loc_mins_celsius, 'time': timepoints})
    print('loc yearly temp is extracted, writing to file')
    loc_temp_table.to_csv(output_file, index=False)
    return loc_temp_table

In [ ]:
urb_all_2023 = get_annual_all(['gribfiles/USA2013-2023_land/2023dec.grib','gribfiles/USA2013-2023_land/2023jan.grib',
                               'gribfiles/USA2013-2023_land/2023feb.grib'], 'urb_all_2023.csv')

In [ ]:
'''Calculate the minimum temp (based on hourly data) for a year or any time range, and write it as a table
ALL FILES MUST HAVE THE SAME LOCATION ORDER'''
# All files in the file list must have the same location orders !!!! (when downloading from ERA5 website, the restriction of geographical range must be the same)
# grbs_filename_list is a list of file path/name. output_file is the output table filename. See find_location_indices() for other parameters
# min_temp_window is the number of hours used to calculate the min. default: 0 (just use the minimum datapoint as min)
def get_annual_minimal(grbs_filename_list, output_file, lat_min = 24, lat_max = 50, lon_min = -125, lon_max= -66, convert_long3 = False, min_temp_window = 0):
    grbs_list = []
    for grbs_filename in grbs_filename_list:
        print('opening file' + grbs_filename + '...')
        grbs = pygrib.open(grbs_filename); grbs_list.append(grbs)
        print(grbs_filename + 'is opened')
    print('opened all grbs file required for the minimum calculation, number of files = ' + str(len(grbs_list)))
    print('filtering for locations, retrieving location indices...')
    location_indices, loc_lats, loc_lons = find_location_indices(lat_min, lat_max, lon_min, lon_max, grbs_list[0], convert_long3)
    print('location indices are done, pulling temperature at all time points...')
    loc_all_values, timepoints = retrieve_values_multi_grbs(grbs_list, location_indices)
    print('All time temperatures are found, calculating minimum...')
    yearly_min_temp_table = merge_table(loc_lons, loc_lats, loc_all_values, timepoints, min_temp_window, plot = False) # yearly minimum is calculated in merge_table()
    print('Yearly minimum is calculated, writing to file')
    yearly_min_temp_table.to_csv(output_file, index=False)
    return yearly_min_temp_table

In [ ]:
min_2023 = get_annual_minimal(['gribfiles/USA2013-2023/2023dec.grib','gribfiles/USA2013-2023/2023jan.grib','gribfiles/USA2013-2023/2023feb.grib'], 'min_2023.csv')

In [ ]:
def get_multiple_years_loc(start_year, end_year, folder = 'gribfiles/USA2013-2023/', name_prefix = 'urb_all'):
    yearly_all_df_list = []
    for year_int in range(start_year, end_year + 1):
        print('processing year ' + str(year_int) + ' ...')
        start_time = time.time()
        file_dec = folder + str(year_int) + 'dec.grib'
        file_jan = folder + str(year_int) + 'jan.grib'
        file_feb = folder + str(year_int) + 'feb.grib'
        file_out = name_prefix + str(year_int) + '.csv'
        yearly_all_df = get_annual_all([file_dec, file_jan, file_feb], file_out)
        yearly_all_df_list.append(yearly_all_df)
        end_time = time.time(); elapsed_time = end_time - start_time
        print('finished processing year ' + str(year_int) + 'starting the next...')
        print("Elapsed time:", elapsed_time, "seconds")
    return yearly_all_df_list

In [ ]:
spn_lvl1_2008_23 = get_multiple_years_loc(2010, 2012, folder = 'gribfiles/depth level 1/USA land/', name_prefix='DIX_lvl1_')
#spn_lvl1_2008_23 = get_multiple_years_loc(2010, 2012, folder = 'gribfiles/USA1993-2012_land/', name_prefix='DIX_lvl2_')

In [ ]:
urb_1993_12 = get_multiple_years_loc(1993, 2012, folder = 'gribfiles/USA1993-2012_land/')

In [ ]:
def cal_multiple_years_minimum(start_year, end_year, folder = 'gribfiles/USA2013-2023/', folder_out = ''):
    yearly_min_df_list = []
    for year_int in range(start_year, end_year + 1):
        print('processing year ' + str(year_int) + ' ...')
        start_time = time.time()
        file_dec = folder + str(year_int) + 'dec.grib'
        file_jan = folder + str(year_int) + 'jan.grib'
        file_feb = folder + str(year_int) + 'feb.grib'
        file_out = folder_out + 'min_' + str(year_int) + '.csv'
        yearly_min_df = get_annual_minimal([file_dec, file_jan, file_feb], file_out)
        yearly_min_df_list.append(yearly_min_df)
        end_time = time.time(); elapsed_time = end_time - start_time
        print('finished processing year ' + str(year_int) + 'starting the next...')
        print("Elapsed time:", elapsed_time, "seconds")
    return yearly_min_df_list

In [ ]:
min_1993_12 = cal_multiple_years_minimum(1993, 2012, folder = 'gribfiles/USA1993-2012/')

In [ ]:
'''For ERA regular data where land temp and ocean temp are mixed'''
def cal_multiple_years_minimum_2(start_year, end_year, folder = 'gribfiles/EU 1993-2023/',
                                lat_min = 24, lat_max = 50, lon_min = -125, lon_max= -66, min_temp_window = 0, folder_out = ''):
    yearly_min_df_list = []
    for year_int in range(start_year, end_year + 1):
        print('processing year ' + str(year_int) + ' ...')
        start_time = time.time()
        file_year = folder + str(year_int)[2:4] + 'DJF.grib'
        file_out = folder_out + 'min_' + str(year_int) + '.csv'
        yearly_min_df = get_annual_minimal([file_year], file_out, lat_min = lat_min, lat_max = lat_max, lon_min = lon_min, lon_max = lon_max, min_temp_window = min_temp_window)
        yearly_min_df_list.append(yearly_min_df)
        end_time = time.time(); elapsed_time = end_time - start_time
        print('finished processing year ' + str(year_int) + 'starting the next...')
        print("Elapsed time:", elapsed_time, "seconds")
    return yearly_min_df_list

In [ ]:
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2013, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 168,
                                           folder_out='yearly_minimum/NA/NA_168h_mean/')
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2013, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 720,
                                           folder_out='yearly_minimum/NA/NA_720h_mean/')

In [ ]:
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 1998, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70, min_temp_window = 24, 
                                            folder_out = 'yearly_minimum/EU/EU_24h_mean/')
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70, min_temp_window = 72, 
                                            folder_out = 'yearly_minimum/EU/EU_72h_mean/')

In [ ]:
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70, min_temp_window = 120, 
                                            folder_out = 'yearly_minimum/EU/EU_120h_mean/')
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70, min_temp_window = 168, 
                                            folder_out = 'yearly_minimum/EU/EU_168h_mean/')
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70, min_temp_window = 720, 
                                            folder_out = 'yearly_minimum/EU/EU_720h_mean/')

In [ ]:
as_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/East Asia 1993-2023/', 10, 60, 80, 150, min_temp_window = 168, 
                                            folder_out = 'yearly_minimum/EA/EA_168h_mean/')
as_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/East Asia 1993-2023/', 10, 60, 80, 150, min_temp_window = 720, 
                                            folder_out = 'yearly_minimum/EA/EA_720h_mean/')

In [ ]:
as_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/East Asia 1993-2023/', 10, 60, 80, 150, min_temp_window = 72, 
                                            folder_out = 'yearly_minimum/EA/EA_72h_median/')
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 24,
                                           folder_out = 'yearly_minimum/NA/NA_24h_median/')
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 72,
                                           folder_out = 'yearly_minimum/NA/NA_72h_median/')
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 120,
                                           folder_out = 'yearly_minimum/NA/NA_120h_median/') 
# median/mean switch is in merge_table() function

In [ ]:
#na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 3,
 #                                          folder_out='yearly_minimum/NA/NA_3h_mean/')
#na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 6,
 #                                          folder_out='yearly_minimum/NA/NA_6h_mean/')
na_min_93_23 = cal_multiple_years_minimum_2(2005, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 15,
                                           folder_out='yearly_minimum/NA/NA_15h_mean/')

In [ ]:
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45, min_temp_window = 24)

In [ ]:
eu_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/EU 1993-2023/', 30, 70, -30, 70)

In [ ]:
as_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/East Asia 1993-2023/', 10, 60, 80, 150)

In [ ]:
na_min_93_23 = cal_multiple_years_minimum_2(1993, 2023, 'gribfiles/NA 1993-2023/', 20, 70, -150, -45)

In [ ]:
min_2021_22 = cal_multiple_years_minimum(2015, 2020)

In [ ]:
# Merge all yearly minimum files into one compiled file #
def merge_yearly_minimum(start_year, end_year, folder = 'yearly_minimum/EU/', filename = "yearly_minimum_30_year_compile.csv"):
    dfs = []
    for year_int in range(start_year, end_year + 1):
         yearly_df = pd.read_csv(folder + 'min_' + str(year_int) + '.csv')
         yearly_df.rename(columns={'min_temp': 'min_temp_' + str(year_int)}, inplace=True)
         yearly_df.rename(columns={'time': 'min_temp_timepoints_' + str(year_int)}, inplace=True)
         dfs = dfs + [yearly_df]
    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=['latitude', 'longitude'])
    print(merged_df.shape)
    column_names = merged_df.columns.tolist(); print(column_names)
    merged_df.to_csv(folder + filename, index = False)
    return(merged_df)

In [ ]:
#yearly_minimum_30_year_compile_EU = merge_yearly_minimum(1993,2023, folder = 'yearly_minimum/EU/EU_24h_mean/', 
 #                                                     filename = 'yearly_24hmean_minimum_30_year_compile_EU.csv')
#yearly_minimum_30_year_compile_EA = merge_yearly_minimum(1993,2023, folder = 'yearly_minimum/EA/EA_168h_mean/', 
 #                                                    filename = 'yearly_168hmean_minimum_30_year_compile_EA.csv')
yearly_minimum_30_year_compile_NA = merge_yearly_minimum(1993,2023, folder = 'yearly_minimum/NA/NA_72h_mean/', 
                                                      filename = 'yearly_72hmean_minimum_30_year_compile_NA.csv')

In [ ]:
''' Calculate multiple stats based on 30 years yearly minimum '''
def process_merged_yearly_minimum(file_path):
    # Load data
    df = pd.read_csv(file_path)
    cols_10_years = ['min_temp_' + str(year) for year in range(2014, 2024)]
    cols_15_years = ['min_temp_' + str(year) for year in range(2009, 2024)]
    cols_20_years = ['min_temp_' + str(year) for year in range(2004, 2024)]
    cols_30_years = ['min_temp_' + str(year) for year in range(1994, 2024)]    
    # Calculate averages and minimums
    df['recent_10_year_averaged_min'] = df[cols_10_years].mean(axis=1)
    df['recent_10_year_absolute_min'] = df[cols_10_years].min(axis=1)
    df['recent_15_year_averaged_min'] = df[cols_15_years].mean(axis=1)
    df['recent_15_year_absolute_min'] = df[cols_15_years].min(axis=1)
    df['recent_20_year_averaged_min'] = df[cols_20_years].mean(axis=1)
    df['recent_20_year_absolute_min'] = df[cols_20_years].min(axis=1)
    df['recent_30_year_averaged_min'] = df[cols_30_years].mean(axis=1)
    df['recent_30_year_absolute_min'] = df[cols_30_years].min(axis=1)  
    # Calculate percentilesfig.suptitle('Overall Title for the Composite Figure', fontsize=16)
    for years, cols in [('10', cols_10_years), ('15', cols_15_years), ('20', cols_20_years), ('30', cols_30_years)]:
        df[f'recent_{years}_year_5th_percentile'] = df[cols].quantile(0.05, axis=1) # colname format: 'recent_10_year_5th_percentile'
        df[f'recent_{years}_year_10th_percentile'] = df[cols].quantile(0.1, axis=1)
        df[f'recent_{years}_year_25th_percentile'] = df[cols].quantile(0.25, axis=1)
        df[f'recent_{years}_year_50th_percentile'] = df[cols].quantile(0.5, axis=1)
    output_file_path = file_path.replace('.csv', '_with_cal.csv')
    df.to_csv(output_file_path, index=False); print("calulation is done for " + file_path)
    return df

In [ ]:
#cal_720h_EU = process_merged_yearly_minimum('yearly_minimum/EU/EU_24h_mean/yearly_24hmean_minimum_30_year_compile_EU.csv')
cal_72hmean_NA = process_merged_yearly_minimum('yearly_minimum/NA/NA_72h_mean/yearly_72hmean_minimum_30_year_compile_NA.csv')
#cal_720hmean_EA = process_merged_yearly_minimum('yearly_minimum/EA/EA_720h_mean/yearly_720hmean_minimum_30_year_compile_EA.csv')
#cal_NA = process_merged_yearly_minimum('yearly_minimum/NA/yearly_minimum_30_year_compile_NA.csv')

In [ ]:
cal_EU = process_merged_yearly_minimum('yearly_minimum/EU/yearly_minimum_30_year_compile_EU.csv')
cal_EA = process_merged_yearly_minimum('yearly_minimum/EA/yearly_minimum_30_year_compile_EA.csv')
cal_NA = process_merged_yearly_minimum('yearly_minimum/NA/yearly_minimum_30_year_compile_NA.csv')

In [ ]:
'''Filter for locations of interests'''
# Helper function for calculating yearly minimum from grib files
# since the grib file include all corrds on the globe. first we only extract the area of interests
# longitude inputs are in long1 (-180-180) but the grib file is in long3 (0-360) (only for whole region data. if sub-region extraction, longitude inputs are in long1)
def find_location_indices(lat_min, lat_max, lon_min, lon_max, para_grbs, convert_long3 = False):
    grb1 = para_grbs[1] # grb message numbers start at 1
    print("read in the first entry/message of grbs")
    lats, lons = grb1.latlons()
    lats = lats.flatten(); lons = lons.flatten()
    print("retrieved lat and lons"); print("total locations in the file is " + str(len(lats)))
    if (convert_long3): lon_min = (lon_min + 360) % 360; lon_max = (lon_max + 360) % 360 # only for full region dataset
    location_indices = []
    for i in range(len(lats)):
        if lat_min <= lats[i] <= lat_max and lon_min <= lons[i] <= lon_max:
            location_indices.append(i)
    loc_lats = lats[location_indices]; loc_lons = lons[location_indices]
    return location_indices, loc_lats, loc_lons

In [ ]:
# test
location_indices, loc_lats, loc_lons = find_location_indices(24, 50, -125, -66, jan20, convert_long3 = False)
print('Number of locations coords found = ' + str(len(location_indices)))
print('head of lats = ' + str(loc_lats[1:15])); print('head of lons = ' + str(loc_lons[1:15]))

In [ ]:
'''Read in all temperature data from multiple grb files with the same range (the same location indices will get the same locations)'''
# Helper function for calculating yearly minimum from grib files
def retrieve_values_multi_grbs(grbs_list, location_indices):
    print("number of locations = " + str(len(location_indices)))
    res_vals = np.empty((0, len(location_indices))) # res_vals will be a 2d array with first dimension as filtered locations, second dimensian as timepoints
    timepoints = np.array([])
    count = 0; file_count = 0
    for grbs in grbs_list:
        for grb in grbs: # grb is 1 message entry in a gribs file
            loc_values = grb.values.flatten()[location_indices] # when unmask grb.values, fill values for unvalid data/NAs are 9999
            res_vals = np.vstack((res_vals, loc_values))
            timepoints = np.append(timepoints, grb.validDate)
            count = count + 1
        file_count = file_count + 1
        print('finished processing ' + str(file_count) + '/' + str(len(grbs_list)) + ' grb file in the list, starting the next...')
    print("Total number of grb files read is " + str(file_count))
    print("Total number of grb message read is " + str(count))
    print(res_vals.shape); print("Total number of timepoints (hourly) are " + str(len(timepoints)))
    return res_vals, timepoints

In [ ]:
loc_all_values, timepoints = retrieve_values_multi_grbs([dec20, jan20], location_indices)

In [ ]:
'''Read in all temperature data from all grb entries (messages/hourly timepoints)'''
# Helper function for calculating yearly minimum from grib files
def retrieve_values(grbs, location_indices):
    print("number of locations = " + str(len(location_indices)))
    res_vals = np.empty((0, len(location_indices)))
    count = 0
    for grb in grbs:
        loc_values = grb.values.flatten()[location_indices] # when unmask grb.values, fill values for unvalid data/NAs are 9999
        res_vals = np.vstack((res_vals, loc_values))
        count = count + 1
    print("Total number of grb message read is " + str(count))
    print(res_vals.shape)
    return res_vals

In [ ]:
loc_all_values = retrieve_values(jan20, location_indices)

In [ ]:
# Helper function for calculating yearly minimum from grib files
def merge_table(loc_lons, loc_lats, loc_all_values, timepoints_list, min_temp_window_size = 1, plot = True ):
    '''Calculate the min temperature in Celsius (by hourly data)''' # NAs are filled by 9999, which doesn't affect the min
    min_indices = np.argmin(loc_all_values, axis = 0)
    print("number of min_indices = " + str(len(min_indices))); print(min_indices)
    loc_timewindow_avg_celsius = np.empty(len(min_indices))
    for column_idx, idx in enumerate(min_indices):
        start = max(0, idx - min_temp_window_size // 2)  # get the start indices of the time window - avoid out of range
        end = min(loc_all_values.shape[0], idx + min_temp_window_size // 2) # get the end indices of the time window
        #Calculate mean for the window and convert from Kelvin to Celsius
        avg_kelvin = np.mean(loc_all_values[start:end, column_idx], axis=0); #print("rule of calculate window min is mean")
        # calculate the median in the [window size] hours flanking the coldest timepoint
        #avg_kelvin = np.median(loc_all_values[start:end, column_idx], axis=0) ; #print("rule of calculate window min is median")
        avg_celsius = avg_kelvin - 273.15
        loc_timewindow_avg_celsius[column_idx] = avg_celsius
    print(loc_timewindow_avg_celsius)
    #loc_hourly_avg_celsius = np.array(loc_hourly_avg_celsius).T  # Transpose back to original shape
    if (plot): plt.hist(min_indices, bins = 30); plt.title('distribution of min temp time')
    min_times = timepoints_list[min_indices] # get timepoints when the min temperature occurred
    '''Mask NAs from the table'''
    na_indices = np.where(loc_timewindow_avg_celsius > 1000) # NAs were filled by 9999 in k unit, then 9999-273.5 in c unit
    mask = np.ones(len(loc_lons), dtype=bool)
    mask[na_indices] = False
    loc_lons_masked = loc_lons[mask]
    loc_lats_masked = loc_lats[mask]
    loc_timewindow_avg_celsius_masked = loc_timewindow_avg_celsius[mask]
    print(type(min_times)); print(type(loc_lons))
    min_times_masked = min_times[mask]
    if (plot): plt.hist(loc_timewindow_avg_celsius_masked, bins = 30); plt.title('distribution of min temp')
    print(len(min_times_masked)); print(len(loc_lats_masked))
    df = pd.DataFrame({'latitude': loc_lats_masked, 'longitude': loc_lons_masked, 'min_temp': loc_timewindow_avg_celsius, 'time': min_times_masked})
    print(df)
    return df

In [ ]:
# Helper function for calculating yearly minimum from grib files
def merge_table_original(loc_lons, loc_lats, loc_all_values, timepoints_list, plot = True):
    '''Calculate the min temperature in Celsius (by hourly data)'''
    loc_hourly_mins_kelvin = np.min(loc_all_values, axis = 0) # since NAs were filled by 9999, doesn't affect the min
    loc_hourly_mins_celsius = loc_hourly_mins_kelvin - 273.15
    min_indices = np.argmin(loc_all_values, axis = 0)
    print(len(min_indices));print(min_indices)
    if (plot): plt.hist(min_indices, bins = 30); plt.title('distribution of min temp time')
    min_times = timepoints_list[min_indices] # get timepoints when the min temperature occurred
    '''Mask NAs from the table'''
    na_indices = np.where(loc_hourly_mins_celsius > 1000) # NAs were filled by 9999 in k unit, then 9999-273.5 in c unit
    mask = np.ones(len(loc_lons), dtype=bool)
    mask[na_indices] = False
    loc_lons_masked = loc_lons[mask]
    loc_lats_masked = loc_lats[mask]
    loc_hourly_mins_celsius_masked = loc_hourly_mins_celsius[mask]
    print(type(min_times)); print(type(loc_lons))
    min_times_masked = min_times[mask]
    if (plot): plt.hist(loc_hourly_mins_celsius_masked, bins = 30); plt.title('distribution of min temp')
    print(len(min_times_masked)); print(len(loc_lats_masked))
    df = pd.DataFrame({'latitude': loc_lats_masked, 'longitude': loc_lons_masked, 'min_temp': loc_hourly_mins_celsius_masked, 'time': min_times_masked})
    print(df)
    return df

In [ ]:
min_table = merge_table(loc_lons, loc_lats, loc_all_values, timepoints)
min_table.to_csv("soil_lvl2_min_temp_2020", index=False)

In [ ]:
location_indices_asia, loc_lats_asia, loc_lons_asia = find_location_indices(15, 50, 85, 150, grbs, convert_long3 = True)
loc_all_values_asia, timepoints_asia = retrieve_values_multi_grbs([grbs], location_indices_asia)

In [ ]:
min_table_asia = merge_table(loc_lons_asia, loc_lats_asia, loc_all_values_asia, timepoints_asia)
min_table_asia.to_csv("soil_asia_min_temp_jan_2023", index=False)

In [ ]:
yearly_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/yearly_minimum_30_year_compile_NA_with_cal.csv')
yearly_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/yearly_minimum_30_year_compile_EA_with_cal.csv')
yearly_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/yearly_minimum_30_year_compile_EU_with_cal.csv')

In [ ]:
yearly_24hmean_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/NA_24h_mean/yearly_24hmean_minimum_30_year_compile_NA_with_cal.csv')
yearly_72hmean_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/NA_72h_mean/yearly_72hmean_minimum_30_year_compile_NA_with_cal.csv')
yearly_120hmean_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/NA_120h_mean/yearly_120hmean_minimum_30_year_compile_NA_with_cal.csv')
yearly_168hmean_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/NA_168h_mean/yearly_168hmean_minimum_30_year_compile_NA_with_cal.csv')                                  
yearly_720hmean_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/NA_720h_mean/yearly_720hmean_minimum_30_year_compile_NA_with_cal.csv')

In [ ]:
yearly_24hmean_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/EU_24h_mean/yearly_24hmean_minimum_30_year_compile_EU_with_cal.csv')
yearly_72hmean_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/EU_72h_mean/yearly_72hmean_minimum_30_year_compile_EU_with_cal.csv')
yearly_120hmean_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/EU_120h_mean/yearly_120hmean_minimum_30_year_compile_EU_with_cal.csv')
yearly_168hmean_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/EU_168h_mean/yearly_168hmean_minimum_30_year_compile_EU_with_cal.csv')
yearly_720hmean_minimum_30_year_cal_EU = pd.read_csv('yearly_minimum/EU/EU_720h_mean/yearly_720hmean_minimum_30_year_compile_EU_with_cal.csv')

In [ ]:
yearly_24hmean_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/EA_24h_mean/yearly_24hmean_minimum_30_year_compile_EA_with_cal.csv')
yearly_72hmean_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/EA_72h_mean/yearly_72hmean_minimum_30_year_compile_EA_with_cal.csv')
yearly_120hmean_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/EA_120h_mean/yearly_120hmean_minimum_30_year_compile_EA_with_cal.csv')
yearly_168hmean_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/EA_168h_mean/yearly_168hmean_minimum_30_year_compile_EA_with_cal.csv')
yearly_720hmean_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/EA_720h_mean/yearly_720hmean_minimum_30_year_compile_EA_with_cal.csv')

In [ ]:
'''Helper functions for plotting'''
def rect_from_bound(xmin, xmax, ymin, ymax):
    """Returns list of (x,y)'s for a rectangle"""
    xs = [xmax, xmin, xmin, xmax, xmax]
    ys = [ymax, ymax, ymin, ymin, ymax]
    return [(x, y) for x, y in zip(xs, ys)]

# request data for use by geopandas
resolution = '10m'
category = 'cultural'
name = 'admin_0_countries'

shpfilename = shapereader.natural_earth(resolution, category, name)
df = geopandas.read_file(shpfilename)

In [ ]:
'''Plotting prepartion'''
# Make an identical temp color scale for every figure
usda_hardinesszone_colors = OrderedDict([
    ('1a', '#d6d6ff'), ('1b', '#c4c4f2'), ('2a', '#ababd9'), ('2b', '#ebb0eb'),
    ('3a', '#e091eb'), ('3b', '#cf7ddb'), ('4a', '#a66bff'), ('4b', '#5a75ed'),
    ('5a', '#73a1ff'), ('5b', '#5ec9e0'), ('6a', '#47ba47'), ('6b', '#78c756'),
    ('7a', '#abd669'), ('7b', '#cddb70'), ('8a', '#edda85'), ('8b', '#ebcb57'),
    ('9a', '#dbb64f'), ('9b', '#f5b678'), ('10a', '#eb9c36'), ('10b', '#e6781e'),
    ('11a', '#e6561e'), ('11b', '#e88564'), ('12a', '#d4594e'), ('12b', '#b51228'),
    ('13a', '#962f1d'), ('13b', '#751a00')
]) # this is 26 colors
usda_cmap = ListedColormap(list(usda_hardinesszone_colors.values()))
values = np.arange(0, len(usda_hardinesszone_colors))
boundaries = np.arange(-15, len(usda_hardinesszone_colors) - 15) 
print(boundaries)
norm = BoundaryNorm(boundaries, len(usda_hardinesszone_colors))
# get geometry of a country
poly = [df.loc[df['ADMIN'] == 'United States of America']['geometry'].values[0]]
stamen_terrain = cimgt.Stamen('terrain-background')

In [ ]:
"""Generate a light blue/light purple color based on temperature."""
def generate_color(temp):
    # Map temperature range from -40°C to -16°C to a value between 0 and 1
    normalized_temp = (temp + 40) / 24  # Normalize temperature to range [0, 1]
    # Calculate hue for the color (blue to purple)
    hue = 240 - (normalized_temp * 10)  # Hue ranges from 240 (blue) to 120 (purple)
    # Set saturation and value (brightness) to create a light color
    saturation = 1-normalized_temp  # Low saturation
    value = normalized_temp*0.8 + 0.2  # High value (brightness)   
    rgb = tuple(int(i * 255) for i in mcolors.hsv_to_rgb([hue/360, saturation, value])) # Convert HSV to RGB
    # Format as hexadecimal color string
    return '#{:02x}{:02x}{:02x}'.format(*rgb)
extended_temp_colors = OrderedDict()
# Generate colors for temperatures from -40°C to -16°C
for temp in range(-40, -15):
    key = str(temp)  # Convert temperature to string for the key
    color = generate_color(temp)  # Generate color based on temperature
    extended_temp_colors[key] = color
print("extended number of colors = " + str(len(extended_temp_colors)))

full_temp_colors = OrderedDict(); full_temp_colors.update(extended_temp_colors); full_temp_colors.update(usda_hardinesszone_colors)
print(full_temp_colors)
full_temp_cmap = ListedColormap(list(full_temp_colors.values())); print("total number of colors = " + str(len(full_temp_colors)))
# show the palette in barplot
test_temps, test_colors = zip(*full_temp_colors.items())
fig, ax = plt.subplots(figsize=(10, 2))
ax.bar(test_temps, len(test_temps) * [1], color=test_colors); ax.set_xticklabels(test_temps, rotation=90)
ax.set_yticks([]) ; ax.set_title('Extended Temperature Color Palette')
plt.show()

In [ ]:
'''Create color scale for survival percent map'''
# Define a function to adjust brightness of colors
def adjust_brightness(color, brightness_factor):
    """Adjust the brightness of a color represented as an RGB tuple."""
    hsv = mcolors.rgb_to_hsv(color[:3])
    hsv[2] = max(0, min(1, hsv[2] * brightness_factor))
    rgb = mcolors.hsv_to_rgb(hsv)
    return rgb
make_survival_colorscale = OrderedDict()
# Generate colors from the Viridis color scale
for i in range(21):  # 21 steps from 0 to 1
    key = round(i * 0.05, 2)  # Round the key to two decimal places
    color = cm.viridis(i / 19)  # Get the color from the Viridis color scale 
    adjusted_color = adjust_brightness(color, 20) # Adjust brightness, increase by 20%
    value = '#{0:02x}{1:02x}{2:02x}'.format(int(adjusted_color[0] * 255), int(adjusted_color[1] * 255), int(adjusted_color[2] * 255))
    make_survival_colorscale[key] = value
print(make_survival_colorscale)
#survival_scale_colors = OrderedDict([(0.0, '#510164'), (0.05, '#551679'), (0.1, '#562b8c'), (0.15, '#533e99'), (0.2, '#4d51a2'), (0.25, '#4662a6'), (0.3, '#3f71a9'), (0.35, '#3881aa'), (0.4, '#3290aa'), (0.45, '#2c9eaa'), (0.5, '#27ada8'), (0.55, '#24bba4'), (0.6, '#29c99e'), (0.65, '#38d794'), (0.7, '#51e586'), (0.75, '#70f175'), (0.8, '#92fb61'), (0.85, '#b5ff46'), (0.9, '#d8ff2b'), (0.95, '#faff1b'), (1.0, '#ffe824')])
survival_scale_colors = OrderedDict([(0.0, '#ce03ff'), (0.05, '#b32fff'), (0.1, '#9d4eff'), (0.15, '#8a67ff'), (0.2, '#7a7fff'), (0.25, '#6b96ff'), (0.3, '#5fabff'), (0.35, '#54c1ff'), (0.4, '#4ad7ff'), (0.45, '#42edff'), (0.5, '#39fff7'), (0.55, '#31ffe0'), (0.6, '#34ffc8'), (0.7, '#5aff96'), (0.75, '#77ff7b'), (0.8, '#94ff62'), (0.85, '#b5ff46'), (0.9, '#d8ff2b'), (0.95, '#faff1b'), (1.0, '#ffe824')])
survival_cmap = ListedColormap(list(survival_scale_colors.values()))
sc_values = np.arange(0, len(survival_scale_colors))
sc_boundaries = np.arange(0, len(survival_scale_colors) - 0) 
print(boundaries)
sc_norm = BoundaryNorm(sc_boundaries, len(survival_scale_colors))

In [ ]:
msi_colors = {'N Japan Msi': ['#1864ec', '#6495ed', '#8cb2f6'], 
              'C Japan Msi': ['#00ff00', '#66ff66', '#b3ffb3'], 'S Japan Msi': ['#ffff00', '#ffff66', '#ffffb3'],
             'Korea, N China Msi': ['#ff0000', '#ff3333', '#ff8080'], 'Sichuan Msi': ['#ffa500', '#ffc04d', '#ffd280'], 
             'Yangtze-Qinling Msi': ['#006400', '#4d934d', '#99c199'], 'SE China Msi': ['#7800b0', '#cd98ff', '#ebd6ff']}
msa_colors = {'N China 2x Msa': ['#ffa500', '#ffc04d', '#ffd280'], 'NE China/Korea/Russia 2x Msa': ['#7800b0', '#cd98ff', '#ebd6ff'], 
              'S Japan 4x Msa': ['#Ea6b80', '#ffc0cb', '#f7c4cc'], 'N Japan 4x Msa': ['#0000ff', '#4d4dff', '#9999ff'],
             'N China/Korea/Russia 4x Msa': ['#ff0000', '#ff3333', '#ff8080'], 
              'Yangtze 2x (ssp. lutarioriparius) Msa': ['#00ff00', '#66ff66', '#b3ffb3']}

In [ ]:
yearly_minimum_30_year_compile = pd.read_csv('yearly_minimum/USA/yearly_minimum_30_year_compile_with_cal.csv')
lt_all_geno = pd.read_csv('ltmulti2_pred_2y_avg_cleaned.txt', sep='\t', encoding='latin1')  # Use sep='\t' for tab-delimited files
def plot_species(species, color_dict, lt_table = lt_all_geno, 
                temp_table = yearly_minimum_30_year_compile, temp_column = 'recent_30_year_absolute_min', sel_entry = None, USonly = False):
    if (sel_entry is not None): species_df = lt_table.loc[lt_table['entry'] in sel_entry]
    else: species_df = lt_table.loc[lt_table['type'] == species]
    for i in range(species_df.shape[0]):
        lt5 = species_df['lt5_pred2'].iloc[i]; lt10 = species_df['lt10_pred2'].iloc[i]
        entry_color = ['#343431', '#797a73', '#adaea4'] if color_dict is None else color_dict.get(species_df['genetic_group'].iloc[i])
        plot_accession_adaptation(temp_table, lt10, lt5, lt5, temp_column, entry_color,
        title=species_df['entry'].iloc[i] + ', LT₅ = ' + str(lt5) + ' ℃ , LT₁₀ = ' + str(lt10) + ' ℃ ' + temp_column, USonly=USonly)

    pass
print(lt_all_geno)

In [ ]:
plot_species('msa', color_dict = msa_colors)

In [ ]:
plot_species('mxg', color_dict = None, temp_column = 'recent_30_year_absolute_min')

In [ ]:
'''Make a color scaled landscape map of minimum temperature in the area, based on the min_table selected'''
### SINGLE FIGURE ###
def plot_lt50_dist(min_table, column_to_plot, species, color_dict, lat_min, lat_max, long_min, long_max, title = '', mask = True, 
                  show_title = True, show_colorbar = True, ax = None, remove_ocean = False, lt_table = lt_all_geno):
    min_table = min_table[(min_table['latitude'] >= lat_min) & (min_table['latitude'] <= lat_max) &
                       (min_table['longitude'] >= long_min) & (min_table['longitude'] <= long_max)]
    if (ax == None): fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    # Add the scatter plot
    color_norm = Normalize(vmin=-40, vmax=11)    # next line, s = 6 for composite figures
    sc = ax.scatter(min_table['longitude'], min_table['latitude'], c=min_table[column_to_plot], cmap=full_temp_cmap, norm =color_norm, s=16, marker='o', edgecolors='none')
    species_df = lt_table.loc[lt_table['species'] == species]
    for i in range(species_df.shape[0]):
        lon = species_df['longitude'].iloc[i]; lat = species_df['latitude'].iloc[i]
        if pd.isna(lon) or pd.isna(lat):
            continue # Skip if lat/lon is missing
        # Get color from color_dict (default to black if not found)
        group = species_df['genetic_group'].iloc[i]
        point_color = color_dict.get(group, 'black')[0]
        ploidy = species_df['ploidy'].iloc[i]
        marker_shape = 's' if ploidy == 4 else 'o'
        ax.scatter(lon, lat, color=point_color, s=160, marker=marker_shape, edgecolor='black', zorder=10)           # put on top of other layers
    if show_colorbar: 
        cbar = plt.colorbar(sc, ax=ax, label='Temperature in Celsius (°C)', orientation = 'horizontal')
        #cbar.ax.xaxis.set_major_locator(MultipleLocator(10))  # Set major ticks at every 10°C
        #cbar.ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))  # Format major ticks as integers
        cbar.ax.xaxis.set_major_locator(MultipleLocator(5))  # Set minor ticks at every 5°C
        #cbar.ax.xaxis.set_minor_formatter(FormatStrFormatter('%d'))  # Format minor ticks as integers
    if show_title: ax.set_title(title, fontsize = 10)
    gridlines = ax.gridlines(draw_labels=True); gridlines.xlocator = MultipleLocator(10); gridlines.ylocator = MultipleLocator(5)
    gridlines.xlines = False; gridlines.ylines = False
    ax.set_extent([long_min, long_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    # projections that involved
    st_proj = stamen_terrain.crs  #projection used by Stamen images
    ll_proj = ccrs.PlateCarree()  #CRS for raw long/lat
    # Add the United States map
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.LAKES)
    # add OCEAN feature, ref = https://stackoverflow.com/questions/64796914/zooming-in-on-cartopy-map-and-adding-the-ocean-feature-changes-the-entire-plot-t
    if(remove_ocean == False):
        choice = 1
        if choice==1:
            ocean110 = cfeature.NaturalEarthFeature('physical', 'ocean', \
                scale='110m', edgecolor='none', facecolor=cfeature.COLORS['water'])
            ax.add_feature(ocean110)
        else: ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.BORDERS, linestyle='-')
    ax.add_feature(cfeature.STATES, linestyle=':', edgecolor='black')
    ax.add_geometries(poly, crs=ll_proj, facecolor='none', edgecolor='black')
    # make a mask polygon by polygon's difference operation, ref: https://stackoverflow.com/questions/62448828/python-cartopy-map-clip-area-outside-country-polygon
    # https://stackoverflow.com/questions/63936330/python-cartopy-draw-shaded-figure-inside-specific-country
    pad1 = 0.1  #padding, degrees unit
    exts = [poly[0].bounds[0] - pad1, poly[0].bounds[2] + pad1, poly[0].bounds[1] - pad1, poly[0].bounds[3] + pad1];
    # base polygon is a rectangle, another polygon is simplified switzerland
    msk = Polygon(rect_from_bound(*exts)).difference(poly[0].simplify(0.01) )
    msk_stm  = st_proj.project_geometry (msk, ll_proj)  # project geometry to the projection used by stamen
    # plot the mask using semi-transparency (alpha=0.65) on the masked-out portion
    if (mask): ax.add_geometries( msk_stm, st_proj, zorder=12, facecolor='white', edgecolor='none', alpha=1)
    plt.grid(False)
    if (ax == None): plt.show()
pass
#yearly_minimum_30_year_compile = pd.read_csv('yearly_minimum/yearly_minimum_30_year_compile_with_cal.csv')

In [ ]:
plot_lt50_dist(yearly_120hmean_minimum_30_year_cal_EA, 'recent_30_year_50th_percentile', "Msa", msa_colors, 25, 55, 105, 150, '',
              mask = False, show_title = False, show_colorbar = False, remove_ocean = True)
plot_lt50_dist(yearly_120hmean_minimum_30_year_cal_EA, 'recent_30_year_50th_percentile', "Msi", msi_colors, 15, 50, 95, 150, '',
              mask = False, show_title = False, show_colorbar = False, remove_ocean = True)

In [ ]:
'''Make a color scaled landscape map of minimum temperature in the area, based on the min_table selected'''
### SINGLE FIGURE ###
def plot_min_temp(min_table, column_to_plot,  lat_min, lat_max, long_min, long_max, title = '', mask = True, 
                  show_title = True, show_colorbar = True, ax = None, remove_ocean = False):
    min_table = min_table[(min_table['latitude'] >= lat_min) & (min_table['latitude'] <= lat_max) &
                       (min_table['longitude'] >= long_min) & (min_table['longitude'] <= long_max)]
    if (ax == None): fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    # Add the scatter plot
    color_norm = Normalize(vmin=-40, vmax=11)    # next line, s = 6 for composite figures
    sc = ax.scatter(min_table['longitude'], min_table['latitude'], c=min_table[column_to_plot], cmap=full_temp_cmap, norm =color_norm, s=16, marker='o', edgecolors='none')
    if show_colorbar: 
        cbar = plt.colorbar(sc, ax=ax, label='Temperature in Celsius (°C)', orientation = 'horizontal')
        #cbar.ax.xaxis.set_major_locator(MultipleLocator(10))  # Set major ticks at every 10°C
        #cbar.ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))  # Format major ticks as integers
        cbar.ax.xaxis.set_major_locator(MultipleLocator(5))  # Set minor ticks at every 5°C
        #cbar.ax.xaxis.set_minor_formatter(FormatStrFormatter('%d'))  # Format minor ticks as integers
    if show_title: ax.set_title(title, fontsize = 10)
    gridlines = ax.gridlines(draw_labels=True); gridlines.xlocator = MultipleLocator(10); gridlines.ylocator = MultipleLocator(5)
    gridlines.xlines = False; gridlines.ylines = False
    ax.set_extent([long_min, long_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    # projections that involved
    st_proj = stamen_terrain.crs  #projection used by Stamen images
    ll_proj = ccrs.PlateCarree()  #CRS for raw long/lat
    # Add the United States map
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.LAKES)
    # add OCEAN feature, ref = https://stackoverflow.com/questions/64796914/zooming-in-on-cartopy-map-and-adding-the-ocean-feature-changes-the-entire-plot-t
    if(remove_ocean == False):
        choice = 1
        if choice==1:
            ocean110 = cfeature.NaturalEarthFeature('physical', 'ocean', \
                scale='110m', edgecolor='none', facecolor=cfeature.COLORS['water'])
            ax.add_feature(ocean110)
        else: ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.BORDERS, linestyle='-')
    ax.add_feature(cfeature.STATES, linestyle=':', edgecolor='black')
    ax.add_geometries(poly, crs=ll_proj, facecolor='none', edgecolor='black')
    # make a mask polygon by polygon's difference operation, ref: https://stackoverflow.com/questions/62448828/python-cartopy-map-clip-area-outside-country-polygon
    # https://stackoverflow.com/questions/63936330/python-cartopy-draw-shaded-figure-inside-specific-country
    pad1 = 0.1  #padding, degrees unit
    exts = [poly[0].bounds[0] - pad1, poly[0].bounds[2] + pad1, poly[0].bounds[1] - pad1, poly[0].bounds[3] + pad1];
    # base polygon is a rectangle, another polygon is simplified switzerland
    msk = Polygon(rect_from_bound(*exts)).difference(poly[0].simplify(0.01) )
    msk_stm  = st_proj.project_geometry (msk, ll_proj)  # project geometry to the projection used by stamen
    # plot the mask using semi-transparency (alpha=0.65) on the masked-out portion
    if (mask): ax.add_geometries( msk_stm, st_proj, zorder=12, facecolor='white', edgecolor='none', alpha=1)
    plt.grid(False)
    if (ax == None): plt.show()
pass
#yearly_minimum_30_year_compile = pd.read_csv('yearly_minimum/yearly_minimum_30_year_compile_with_cal.csv')

In [ ]:
'''Make a three-levels survival rate map for a given accession (input 3 different LTs) of based on LT values'''
### SINGLE FIGURE ###
def plot_accession_adaptation(min_table, LT10, LT5, LT0, column_for_temp, lat_min, lat_max, long_min, long_max,
            color= ['#343431', '#797a73', '#adaea4'], title = '', mask = True, USonly = False):
    min_table = min_table[(min_table['latitude'] >= lat_min) & (min_table['latitude'] <= lat_max) &
                       (min_table['longitude'] >= long_min) & (min_table['longitude'] <= long_max)]
    fig = plt.figure(figsize=(10, 6))
    # Create a subplot for the United States map
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    # Add the scatter plot
    #sc = ax.scatter(min_table['longitude'], min_table['latitude'], c=min_table[column_for_temp], 
     #               norm = norm, cmap=usda_cmap, s=6, marker='o', edgecolors='none')
    mask_gt_LT10 = min_table[column_for_temp] > LT10
    mask_gt_LT5 = min_table[column_for_temp] > LT5
    mask_gt_LT0 = min_table[column_for_temp] > LT0
    #print(len(min_table['longitude'][mask_gt_LT10])); print(len(min_table['longitude'][mask_gt_LT5])); print(len(min_table['longitude'][mask_gt_LT0]))
    # Plot points based on LTs with different shades. Pick shades, alpha doesn't work (dots overlaps)
    sc = ax.scatter(min_table['longitude'][mask_gt_LT10], min_table['latitude'][mask_gt_LT10], color = color[2], s=6, marker='o')
    plt.scatter(min_table['longitude'][mask_gt_LT5], min_table['latitude'][mask_gt_LT5], color = color[1], s=6, marker='o')
    plt.scatter(min_table['longitude'][mask_gt_LT0], min_table['latitude'][mask_gt_LT0], color = color[0], s=6, marker='o')
    #plt.colorbar(sc, ax=ax, label='Temperature in Celsius (°C)', orientation = 'horizontal')
    ax.set_title(title, fontsize = 10)
    gridlines = ax.gridlines(draw_labels=True)
    ax.set_extent([long_min, long_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    #ax.set_xticks([-130,-120,-110,-100,-90,-80,-70,-60])
    #ax.set_yticks([20,25,30,35,40,45,50])
    # projections that involved
    st_proj = stamen_terrain.crs  #projection used by Stamen images
    ll_proj = ccrs.PlateCarree()  #CRS for raw long/lat
    # Add the United States map
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.LAKES)
    # add OCEAN feature, ref = https://stackoverflow.com/questions/64796914/zooming-in-on-cartopy-map-and-adding-the-ocean-feature-changes-the-entire-plot-t
    choice = 1
    if choice==1:
        ocean110 = cfeature.NaturalEarthFeature('physical', 'ocean', \
            scale='110m', edgecolor='none', facecolor=cfeature.COLORS['water'])
        ax.add_feature(ocean110)
    else: ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.BORDERS, linestyle='-')
    ax.add_feature(cfeature.STATES, linestyle=':', edgecolor='black')
    ax.add_geometries(poly, crs=ll_proj, facecolor='none', edgecolor='black')
    # make a mask polygon by polygon's difference operation, ref: https://stackoverflow.com/questions/62448828/python-cartopy-map-clip-area-outside-country-polygon
    # https://stackoverflow.com/questions/63936330/python-cartopy-draw-shaded-figure-inside-specific-country
    pad1 = 0.1  #padding, degrees unit
    exts = [poly[0].bounds[0] - pad1, poly[0].bounds[2] + pad1, poly[0].bounds[1] - pad1, poly[0].bounds[3] + pad1];
    # base polygon is a rectangle, another polygon is simplified switzerland
    msk = Polygon(rect_from_bound(*exts)).difference( poly[0].simplify(0.01) )
    msk_stm  = st_proj.project_geometry (msk, ll_proj)  # project geometry to the projection used by stamen
    # plot the mask using semi-transparency (alpha=0.65) on the masked-out portion
    if (USonly): ax.add_geometries( msk_stm, st_proj, zorder=12, facecolor='white', edgecolor='none', alpha=1)
    plt.show()

In [ ]:
# test 
# illinois lt5 = -2.09 not displayed
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -5.22, -2.78, -2.78, 'recent_30_year_absolute_min', 23, 52, -130, -60, title = '')
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -5.22, -2.78, -2.78, 'recent_30_year_25th_percentile', 23, 52, -130, -60, title = '')
#plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -5.22, -2.78, -2.78, 'recent_30_year_10th_percentile', 23, 52, -130, -60, title = '')
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -5.22, -2.78, -2.78, 'recent_30_year_50th_percentile', 23, 52, -130, -60, title = '')
# nagara lt5 = -3.55 not displayed
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -7.35, -4.39, -4.39, 'recent_30_year_absolute_min', 23, 52, -130, -60, title = '')
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -7.35, -4.39, -4.39, 'recent_30_year_25th_percentile', 23, 52, -130, -60, title = '')
#plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -7.35, -4.39, -4.39, 'recent_30_year_10th_percentile', 23, 52, -130, -60, title = '')
plot_accession_adaptation(yearly_minimum_30_year_cal_NA, -7.35, -4.39, -4.39, 'recent_30_year_50th_percentile', 23, 52, -130, -60, title = '')

In [ ]:
''' plot a continuous scale of survival rate for a given accession (input slope and intercept of LT50 formula)'''
### SINGLE FIGURE ###
# Need to input the formula of LT50 - temp (slop and intercept): 
# qnorm(survival) = a*temp + b; survival = pnorm(a*temp + b) = pnorm(slope*temp + intercept) 
def plot_accession_adaptation_contsurvival(min_table, column_to_plot,  lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, title = '', 
                                       mask = True, show_title = True, show_colorbar = True, ax = None):
    min_table = min_table[(min_table['latitude'] >= lat_min) & (min_table['latitude'] <= lat_max) &
                       (min_table['longitude'] >= long_min) & (min_table['longitude'] <= long_max)]
    if (ax == None): fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    # Add the scatter plot
    survival_at_loc = snorm.cdf(lt_slope*min_table[column_to_plot] + lt_intercept) # transfer temp to survival based on survival formula
    color_norm = Normalize(vmin=np.min(survival_at_loc), vmax=np.max(survival_at_loc))
    sc = ax.scatter(min_table['longitude'], min_table['latitude'], c=survival_at_loc, norm = color_norm, cmap=survival_cmap, s=6, marker='o', edgecolors='none')
    if show_colorbar: plt.colorbar(sc, ax=ax, label='Predicted Survival', orientation = 'horizontal')
    if show_title: ax.set_title(title, fontsize = 10)
    gridlines = ax.gridlines(draw_labels=True); gridlines.xlocator = MultipleLocator(10); gridlines.ylocator = MultipleLocator(5)
    ax.set_extent([long_min, long_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    # projections that involved
    st_proj = stamen_terrain.crs  #projection used by Stamen images
    ll_proj = ccrs.PlateCarree()  #CRS for raw long/lat
    # Add the United States map
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.LAKES)
    # add OCEAN feature, ref = https://stackoverflow.com/questions/64796914/zooming-in-on-cartopy-map-and-adding-the-ocean-feature-changes-the-entire-plot-t
    choice = 1
    if choice==1:
        ocean110 = cfeature.NaturalEarthFeature('physical', 'ocean', \
            scale='110m', edgecolor='none', facecolor=cfeature.COLORS['water'])
        ax.add_feature(ocean110)
    else: ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.BORDERS, linestyle='-')
    ax.add_feature(cfeature.STATES, linestyle=':', edgecolor='black')
    ax.add_geometries(poly, crs=ll_proj, facecolor='none', edgecolor='black')
    # make a mask polygon by polygon's difference operation, ref: https://stackoverflow.com/questions/62448828/python-cartopy-map-clip-area-outside-country-polygon
    # https://stackoverflow.com/questions/63936330/python-cartopy-draw-shaded-figure-inside-specific-country
    pad1 = 0.1  #padding, degrees unit
    exts = [poly[0].bounds[0] - pad1, poly[0].bounds[2] + pad1, poly[0].bounds[1] - pad1, poly[0].bounds[3] + pad1];
    # base polygon is a rectangle, another polygon is simplified switzerland
    msk = Polygon(rect_from_bound(*exts)).difference(poly[0].simplify(0.01) )
    msk_stm  = st_proj.project_geometry (msk, ll_proj)  # project geometry to the projection used by stamen
    # plot the mask using semi-transparency (alpha=0.65) on the masked-out portion
    if (mask): ax.add_geometries( msk_stm, st_proj, zorder=12, facecolor='white', edgecolor='none', alpha=1)
    if (ax == None): plt.show()
pass

In [ ]:
''' Produce multiple types of composite figures '''
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_50th_percentile', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,1])
    # Subplot in column 2 row 2
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_25th_percentile', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,1])
    # Subplot in column 2 row 3
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_absolute_min', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,1])
    # Subplot in column 3 row 1
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_50th_percentile', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,2])
    # Subplot in column 3 row 2
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_25th_percentile', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,2])
    # Subplot in column 3 row 3
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_absolute_min', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,2])
    # Subplot in column 4 row 1
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,3])
    # Subplot in column 4 row 2
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,3])
    # Subplot in column 4 row 3
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
''' Produce multiple types of composite figures '''
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_area_multi_temp_NA(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1
    plot_min_temp(yearly_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2
    plot_min_temp(yearly_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3
    plot_min_temp(yearly_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,1])
    # Subplot in column 2 row 2
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,1])
    # Subplot in column 2 row 3
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,1])
    # Subplot in column 3 row 1
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,2])
    # Subplot in column 3 row 2
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,2])
    # Subplot in column 3 row 3
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,2])
    # Subplot in column 4 row 1
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,3])
    # Subplot in column 4 row 2
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,3])
    # Subplot in column 4 row 3
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_NA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
''' Produce multiple types of composite figures '''
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_area_multi_temp_EA(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1
    plot_min_temp(yearly_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2
    plot_min_temp(yearly_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3
    plot_min_temp(yearly_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,1])
    # Subplot in column 2 row 2
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,1])
    # Subplot in column 2 row 3
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,1])
    # Subplot in column 3 row 1
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,2])
    # Subplot in column 3 row 2
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,2])
    # Subplot in column 3 row 3
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,2])
    # Subplot in column 4 row 1
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,3])
    # Subplot in column 4 row 2
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,3])
    # Subplot in column 4 row 3
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
''' Produce multiple types of composite figures '''
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_area_multi_temp_EU(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1
    plot_min_temp(yearly_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2
    plot_min_temp(yearly_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3
    plot_min_temp(yearly_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,1])
    # Subplot in column 2 row 2
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,1])
    # Subplot in column 2 row 3
    plot_min_temp(yearly_120hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,1])
    # Subplot in column 3 row 1
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,2])
    # Subplot in column 3 row 2
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,2])
    # Subplot in column 3 row 3
    plot_min_temp(yearly_168hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,2])
    # Subplot in column 4 row 1
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,3])
    # Subplot in column 4 row 2
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,3])
    # Subplot in column 4 row 3
    plot_min_temp(yearly_720hmean_minimum_30_year_cal_EU, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
''' Produce multiple types of composite figures '''
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_same_temp_multi_area(min_table_1, min_table_2, min_table_3, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 3, figsize=(fig_width, fig_height))
    gs = GridSpec(3, 3, figure=fig, width_ratios=[7, 10, 6], height_ratios=[1, 1, 1])
    for i in range(3):
        for j in range(3):
            axs[i, j].axis('off'); axs[i, j] = fig.add_subplot(gs[i, j], projection=ccrs.PlateCarree())
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1 #######################
    plot_min_temp(min_table_3, 'recent_'+ num_years +'_year_50th_percentile', 20, 60, -130, -60, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2
    plot_min_temp(min_table_3, 'recent_'+ num_years +'_year_25th_percentile', 20, 60, -130, -60, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3
    plot_min_temp(min_table_3, 'recent_'+ num_years +'_year_absolute_min', 20, 60, -130, -60, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1 #######################
    plot_min_temp(min_table_2, 'recent_'+ num_years +'_year_50th_percentile', 30, 70, -30, 70, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,1])
    # Subplot in column 2 row 2
    plot_min_temp(min_table_2, 'recent_'+ num_years +'_year_25th_percentile', 30, 70, -30, 70, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,1])
    # Subplot in column 2 row 3
    plot_min_temp(min_table_2, 'recent_'+ num_years +'_year_absolute_min', 30, 70, -30, 70, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,1])
    # Subplot in column 3 row 1 #######################
    plot_min_temp(min_table_1, 'recent_'+ num_years +'_year_50th_percentile', 20, 60, 90, 150, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,2])
    # Subplot in column 3 row 2
    plot_min_temp(min_table_1, 'recent_'+ num_years +'_year_25th_percentile', 20, 60, 90, 150, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,2])
    # Subplot in column 3 row 3
    plot_min_temp(min_table_1, 'recent_'+ num_years +'_year_absolute_min', 20, 60, 90, 150, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,2])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_compare_single_geno(min_table_1, min_table_2, min_table_3, min_table_4, 
            lt_slope, lt_intercept, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1  [0, 0] ######################
    plot_accession_adaptation_contsurvival(min_table_1, 'recent_'+ num_years +'_year_50th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,0])
    # Subplot in column 1 row 2 [1, 0]
    plot_accession_adaptation_contsurvival(min_table_1,  'recent_'+ num_years +'_year_25th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,0])
    # Subplot in column 1 row 3 [2, 0]
    plot_accession_adaptation_contsurvival(min_table_1, 'recent_'+ num_years +'_year_absolute_min', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,0])
    # Subplot in column 2 row 1 [0, 1] ######################
    plot_accession_adaptation_contsurvival(min_table_2,  'recent_'+ num_years +'_year_50th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,1])
    # Subplot in column 2 row 2 [1, 1]
    plot_accession_adaptation_contsurvival(min_table_2, 'recent_'+ num_years +'_year_25th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,1])
    # Subplot in column 2 row 3 [2, 1]
    plot_accession_adaptation_contsurvival(min_table_2, 'recent_'+ num_years +'_year_absolute_min', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,1])
    # Subplot in column 3 row 1 [0, 2] ######################
    plot_accession_adaptation_contsurvival(min_table_3,  'recent_'+ num_years +'_year_50th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,2])
    # Subplot in column 3 row 2 [1, 2]
    plot_accession_adaptation_contsurvival(min_table_3,  'recent_'+ num_years +'_year_25th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,2])
    # Subplot in column 3 row 3 [2, 2]
    plot_accession_adaptation_contsurvival(min_table_3, 'recent_'+ num_years +'_year_absolute_min', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,2])
    # Subplot in column 4 row 1 [0, 3] ######################
    plot_accession_adaptation_contsurvival(min_table_4,  'recent_'+ num_years +'_year_50th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,3])
    # Subplot in column 4 row 2 [1, 3] 
    plot_accession_adaptation_contsurvival(min_table_4,  'recent_'+ num_years +'_year_25th_percentile', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,3])
    # Subplot in column 4 row 3 [2, 3] 
    plot_accession_adaptation_contsurvival(min_table_4, 'recent_'+ num_years +'_year_absolute_min', lt_slope, lt_intercept, lat_min, lat_max, long_min, long_max, '', 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_compare_robustus(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1  [0, 0] ######################
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2 [1, 0]
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3 [2, 0]
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1 [0, 1] ######################
    plot_accession_adaptation_contsurvival(yearly_120hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,1])
    # Subplot in column 2 row 2 [1, 1]
    plot_accession_adaptation_contsurvival(yearly_120hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,1])
    # Subplot in column 2 row 3 [2, 1]
    plot_accession_adaptation_contsurvival(yearly_120hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,1])
    # Subplot in column 3 row 1 [0, 2] ######################
    plot_accession_adaptation_contsurvival(yearly_24hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,2])
    # Subplot in column 3 row 2 [1, 2]
    plot_accession_adaptation_contsurvival(yearly_24hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,2])
    # Subplot in column 3 row 3 [2, 2]
    plot_accession_adaptation_contsurvival(yearly_24hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,2])
    # Subplot in column 4 row 1 [0, 3] ######################
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,3])
    # Subplot in column 4 row 2 [1, 3] 
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,3])
    # Subplot in column 4 row 3 [2, 3] 
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
### COMPOSITE FIGURE ### need to load single figure functions first 
def plot_composite_figure_compare_robustus_longterm(min_table, fig_width, fig_height, subplots_wspace, subplots_hspace,
    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfolder = 'results/figures/', outputfile = 'composite_fig.png'):
    #  size is in inch (wide, height). 1 inch is 96 pixels. the original size for each subfigure is 10x6
    fig, axs = plt.subplots(3, 4, figsize=(fig_width, fig_height), subplot_kw={'projection': ccrs.PlateCarree()})
    num_years = str(num_years); show_title = False; show_colorbar = False
    # Subplot in column 1 row 1  [0, 0] ######################
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[0,0])
    # Subplot in column 1 row 2 [1, 0]
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[1,0])
    # Subplot in column 1 row 3 [2, 0]
    plot_min_temp(min_table, 'recent_'+ num_years +'_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent '+ num_years +' Years (ERA5 Data)', 
        mask = False, show_title = show_title, show_colorbar = show_colorbar, ax = axs[2,0])
    # Subplot in column 2 row 1 [0, 1] ######################
    plot_accession_adaptation_contsurvival(yearly_720hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,1])
    # Subplot in column 2 row 2 [1, 1]
    plot_accession_adaptation_contsurvival(yearly_720hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,1])
    # Subplot in column 2 row 3 [2, 1]
    plot_accession_adaptation_contsurvival(yearly_720hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,1])
    # Subplot in column 3 row 1 [0, 2] ######################
    plot_accession_adaptation_contsurvival(yearly_168hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,2])
    # Subplot in column 3 row 2 [1, 2]
    plot_accession_adaptation_contsurvival(yearly_168hmean_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,2])
    # Subplot in column 3 row 3 [2, 2]
    plot_accession_adaptation_contsurvival(yearly_168hmean_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,2])
    # Subplot in column 4 row 1 [0, 3] ######################
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[0,3])
    # Subplot in column 4 row 2 [1, 3] 
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA,  'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[1,3])
    # Subplot in column 4 row 3 [2, 3] 
    plot_accession_adaptation_contsurvival(yearly_minimum_30_year_cal_EA, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", 
            mask = False, show_title = False, show_colorbar = False, ax = axs[2,3])
    fig.subplots_adjust(wspace=subplots_wspace, hspace=subplots_hspace)
    #fig.suptitle('temp', fontsize = 24)
    print('data number of years = ' + num_years)
    fig.savefig(outputfolder + outputfile, dpi=150, bbox_inches='tight', pad_inches=0.1) 
    plt.show()
pass

In [ ]:
plot_composite_figure_compare_robustus_longterm(yearly_720hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 10, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_168h_720h_10yr.png')
plot_composite_figure_compare_robustus_longterm(yearly_720hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_168h_720h_15yr.png')
plot_composite_figure_compare_robustus_longterm(yearly_720hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_168h_720h_30yr.png')

In [ ]:
plot_composite_figure_compare_robustus(yearly_120hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 10, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_24h_120h_10yr.png')
plot_composite_figure_compare_robustus(yearly_120hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_24h_120h_15yr.png')
plot_composite_figure_compare_robustus(yearly_120hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 16, subplots_wspace = 0.2, subplots_hspace = 0.2, 
                    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile = 'robustus_compare_24h_120h_30yr.png')

In [ ]:
plot_composite_figure_compare_illinois(yearly_120hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_24h_120h_10yr.png')
plot_composite_figure_compare_illinois(yearly_120hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_24h_120h_15yr.png')
plot_composite_figure_compare_illinois(yearly_120hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_24h_120h_30yr.png')

In [ ]:
plot_composite_figure_compare_illinois_longterm(yearly_720hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 10, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_168h_720h_10yr.png')
plot_composite_figure_compare_illinois_longterm(yearly_720hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_168h_720h_15yr.png')
plot_composite_figure_compare_illinois_longterm(yearly_720hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
                    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile = 'illinois_compare_168h_720h_30yr.png')

In [ ]:
# SERIES 1: SOIL TEMP + 3 SPECIES SURVIVAL, 30 years and 15 years, 3 areas
'NA 30 YEARS'
plot_composite_figure(yearly_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_hourly_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_120hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_168hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_720hmean_30yr,png', outputfolder= 'results/figures/composite1/')
'NA 15 YEARS'
plot_composite_figure(yearly_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_hourly_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_120hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_168hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, outputfile= 'NA_3geno_720hmean_15yr,png', outputfolder= 'results/figures/composite1/')
'EA 30 YEARS' # (20, 60, 90, 150)
plot_composite_figure(yearly_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_hourly_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_120hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_168hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_720hmean_30yr,png', outputfolder= 'results/figures/composite1/')
'EA 15 YEARS'
plot_composite_figure(yearly_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_hourly_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_120hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_168hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2, 
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, outputfile= 'EA_3geno_720hmean_15yr,png', outputfolder= 'results/figures/composite1/')


In [ ]:
'EU 30 YEARS'
plot_composite_figure(yearly_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_hourly_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_120hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_168hmean_30yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_720hmean_30yr,png', outputfolder= 'results/figures/composite1/')
'EU 15 YEARS'
plot_composite_figure(yearly_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_hourly_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_120hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_120hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_168hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_168hmean_15yr,png', outputfolder= 'results/figures/composite1/')
plot_composite_figure(yearly_720hmean_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05, 
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, outputfile= 'EU_3geno_720hmean_15yr,png', outputfolder= 'results/figures/composite1/')

In [ ]:
# SERIES 2: SOIL TEMP BY DIFF CALCULATIONS IN SINGLE LOCATIONS
plot_composite_figure_area_multi_temp_NA(yearly_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.05, num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfile = 'NA_soiltempcomp_30yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')
plot_composite_figure_area_multi_temp_NA(yearly_minimum_30_year_cal_NA, fig_width = 32, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.05, num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfile = 'NA_soiltempcomp_15yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')

plot_composite_figure_area_multi_temp_EA(yearly_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.2, num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfile = 'EA_soiltempcomp_30yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')
plot_composite_figure_area_multi_temp_EA(yearly_minimum_30_year_cal_EA, fig_width = 32, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.2, num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfile = 'EA_soiltempcomp_15yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')

In [ ]:
plot_composite_figure_area_multi_temp_EU(yearly_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.05, num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfile = 'EU_soiltempcomp_30yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')
plot_composite_figure_area_multi_temp_EU(yearly_minimum_30_year_cal_EU, fig_width = 40, fig_height = 15, 
    subplots_wspace = 0.2, subplots_hspace = 0.05, num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfile = 'EU_soiltempcomp_15yr.png', outputfolder = 'results/figures/composite2_soiltempdiffcal/')

In [ ]:
# SERIES 3: SOIL TEMP BY DIFF CALCULATIONS IN MULTIPLE LOCATIONS
plot_composite_figure_same_temp_multi_area(yearly_minimum_30_year_cal_EA, yearly_minimum_30_year_cal_EU, yearly_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = 'hourly_30years.png')
plot_composite_figure_same_temp_multi_area(yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_120hmean_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '120hmean_30years.png')
plot_composite_figure_same_temp_multi_area(yearly_168hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_168hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '168hmean_30years.png')
plot_composite_figure_same_temp_multi_area(yearly_720hmean_minimum_30_year_cal_EA, 
    yearly_720hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '720hmean_30years.png')

plot_composite_figure_same_temp_multi_area(yearly_minimum_30_year_cal_EA, yearly_minimum_30_year_cal_EU, yearly_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = 'hourly_15years.png')
plot_composite_figure_same_temp_multi_area(yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_120hmean_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '120hmean_15years.png')
plot_composite_figure_same_temp_multi_area(yearly_168hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_168hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '168hmean_15years.png')
plot_composite_figure_same_temp_multi_area(yearly_720hmean_minimum_30_year_cal_EA, 
    yearly_720hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_NA, 
    fig_width = 24, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, outputfolder = 'results/figures/composite3_soiltempmultiloc/', outputfile = '720hmean_15years.png')

In [ ]:
# SERIES 4: SINGLE GENOTYPE COMPARED AT DIFFERENT TEMP CALS BY LOC
'NA 30 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_NA_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_NA_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_NA_30yrs.png')
'NA 15 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_EA_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_EA_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 30, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_EA_30yrs.png')
'EA 30 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_NA_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_NA_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_NA, yearly_120hmean_minimum_30_year_cal_NA, 
    yearly_168hmean_minimum_30_year_cal_NA, yearly_720hmean_minimum_30_year_cal_NA, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = -130, long_max = -60, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_NA_15yrs.png')
'EA 15 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_EA_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_EA_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EA, yearly_120hmean_minimum_30_year_cal_EA, 
    yearly_168hmean_minimum_30_year_cal_EA, yearly_720hmean_minimum_30_year_cal_EA, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 32, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.2,
    num_years = 15, lat_min = 20, lat_max = 60, long_min = 90, long_max = 150, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_EA_15yrs.png')

In [ ]:
'EU 30 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_EU_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_EU_30yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 30, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_EU_30yrs.png')
'EU 15 YEARS'
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.52495008, lt_intercept = 2.7410906, 
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'illinois_EU_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.43203056, lt_intercept = 3.1767847, 
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'nagara_EU_15yrs.png')
plot_composite_figure_compare_single_geno(yearly_minimum_30_year_cal_EU, yearly_120hmean_minimum_30_year_cal_EU, 
    yearly_168hmean_minimum_30_year_cal_EU, yearly_720hmean_minimum_30_year_cal_EU, lt_slope = 0.13160141, lt_intercept = 2.3347621,
    fig_width = 40, fig_height = 15, subplots_wspace = 0.2, subplots_hspace = 0.05,
    num_years = 15, lat_min = 30, lat_max = 70, long_min = -30, long_max = 70, 
    outputfolder = 'results/figures/composite4_singlegenomultical/', outputfile = 'robustus_EU_15yrs.png')

In [ ]:
#lt_slope = 0.52495008; lt_intercept = 2.7410906; temp =  -5.3 # illinois/freedom
lt_slope = 0.43203056; lt_intercept = 3.1767847; temp =  -9 # nagara
lt_slope = 0.43203056; lt_intercept = 3.1767847; temp =  -4.2 # nagara
survival_at_loc = snorm.cdf(lt_slope*temp + lt_intercept)
print("predicted survival = " + str(survival_at_loc))

In [ ]:
''' plot continuous survival rate prediction map for muti years for selected genotypes'''
# MULTIPLE FIGURES #
# selected genotypes: illinois, nagara, robustus
def plot_accession_adaptations(min_table, num_years, lat_min, lat_max, long_min, long_max):
    num_years = str(num_years)
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_absolute_min', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_10th_percentile', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 10th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_25th_percentile', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_50th_percentile', 0.52495008, 2.7410906, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Mxg 'Illinois'", mask = False, show_title = True, show_colorbar = True)

    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_absolute_min', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_10th_percentile', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 10th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_25th_percentile', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_50th_percentile', 0.43203056, 3.1767847, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Mxg 'Nagara'", mask = False, show_title = True, show_colorbar = True)

    #plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_absolute_min', 0.10168508, 3.4530164, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'RU2012-048'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_10th_percentile', 0.10168508, 3.4530164, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 10th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'RU2012-048'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_25th_percentile', 0.10168508, 3.4530164, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'RU2012-048'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table,  'recent_'+ num_years +'_year_50th_percentile', 0.10168508, 3.4530164, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'RU2012-048'", mask = False, show_title = True, show_colorbar = True)

    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_absolute_min', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Absolute Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", mask = False, show_title = True, show_colorbar = True)
    #plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_10th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 10th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_25th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under 25th Percentile Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", mask = False, show_title = True, show_colorbar = True)
    plot_accession_adaptation_contsurvival(min_table, 'recent_'+ num_years +'_year_50th_percentile', 0.13160141,2.3347621, lat_min, lat_max, long_min, long_max, "Predicted % of Survival under Median Annual Minimum Temperature over "+ num_years +" years, Msa 'Robustus'", mask = False, show_title = True, show_colorbar = True)
pass

In [ ]:
#plot_accession_adaptation_survival(yearly_minimum_30_year_cal_NA, -7.35, -4.39, -4.39, 'recent_30_year_absolute_min', 23, 52, -130, -60, title = '')
#yearly_minimum_30_year_cal_NA, 23, 52, -130, -60
plot_accession_adaptations(yearly_minimum_30_year_cal_NA, 10,  20, 60, -130, -60)
plot_accession_adaptations(yearly_minimum_30_year_cal_EA, 10, 20, 60, 90, 150)
plot_accession_adaptations(yearly_minimum_30_year_cal_EU, 10, 30, 70, -30, 70)

plot_accession_adaptations(yearly_minimum_30_year_cal_NA, 15,  20, 60, -130, -60)
plot_accession_adaptations(yearly_minimum_30_year_cal_EA, 15, 20, 60, 90, 150)
plot_accession_adaptations(yearly_minimum_30_year_cal_EU, 15, 30, 70, -30, 70)

plot_accession_adaptations(yearly_minimum_30_year_cal_NA, 30,  20, 60, -130, -60)
plot_accession_adaptations(yearly_minimum_30_year_cal_EA, 30, 20, 60, 90, 150)
plot_accession_adaptations(yearly_minimum_30_year_cal_EU, 30, 30, 70, -30, 70)

In [ ]:
yearly_minimum_30_year_compile = pd.read_csv('yearly_minimum/NA/yearly_minimum_30_year_compile_NA_with_cal.csv')

In [ ]:
def gen_region_plots(yearly_minimum_30_year_compile, lat_min, lat_max, long_min, long_max, show_title = True, show_colorbar = True):
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_10_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent 10 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_10_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum 25th Percentile Temperature at Depth of 7-28 cm over the Recent 10 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_10_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent 10 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)

    plot_min_temp(yearly_minimum_30_year_compile, 'recent_15_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent 15 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_15_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum 25th Percentile Temperature at Depth of 7-28 cm over the Recent 15 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_15_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent 15 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_averaged_min', lat_min, lat_max, long_min, long_max, 'Soil Averaged Annual Minimum Temperature at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_5th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Minimum Temperature 5th Percentile at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_10th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum 10th Percentile Temperature at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum 25th Percentile Temperature at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #plot_min_temp(yearly_minimum_30_year_compile, 'recent_20_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent 20 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    #fig = plt.figure(figsize=(30, 6))
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_30_year_absolute_min', lat_min, lat_max, long_min, long_max, 'Soil Absolute Minimum Temperature at Depth of 7-28 cm over the Recent 30 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_30_year_25th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Temperature 25th Percentile at Depth of 7-28 cm over the Recent 30 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    plot_min_temp(yearly_minimum_30_year_compile, 'recent_30_year_50th_percentile', lat_min, lat_max, long_min, long_max, 'Soil Annual Minimum Median Temperature at Depth of 7-28 cm over the Recent 30 Years (ERA5 Data)', mask = False, show_title = show_title, show_colorbar = show_colorbar)
    pass

In [ ]:
#gen_region_plots(yearly_minimum_30_year_cal_NA, 23, 52, -130, -60, False, False)
#gen_region_plots(yearly_minimum_30_year_cal_NA, 20, 55, -130, -60)
gen_region_plots(yearly_minimum_30_year_cal_NA, 20, 60, -130, -60)
gen_region_plots(yearly_minimum_30_year_cal_EA, 20, 60, 90, 150)
gen_region_plots(yearly_minimum_30_year_cal_EU, 30, 70, -30, 70)

In [ ]:
gen_region_plots(yearly_minimum_30_year_cal_NA, 20, 60, -130, -60)
gen_region_plots(yearly_minimum_30_year_cal_EU, 20, 60, -30, 70)
gen_region_plots(yearly_minimum_30_year_cal_EA, 20, 60, 90, 150)

In [ ]:
'''Retrieve soil temperature by a given long and lat'''
def get_soil_temp_at_closest_loc(min_table, column_for_data,  lat, long):
    soil_temp_reference = min_table
    soil_temp_reference['distance'] = np.sqrt((soil_temp_reference['latitude'] - lat)**2 + 
                                                    (soil_temp_reference['longitude'] - long)**2)
    # Find the index of the minimum distance
    closest_row_index = soil_temp_reference['distance'].idxmin()
    print("nearest loc is " + str(min_table.at[closest_row_index, 'latitude'])+ ", " + str(min_table.at[closest_row_index, "longitude"]))
    return min_table.at[closest_row_index, column_for_data]
def retrieve_soil_temp_for_entry_table(lt_table, min_table, columns_for_data, lat_col = 'latitude', long_col = 'longitude'):
    for column_for_data in columns_for_data:
        print (column_for_data)
        for index, row in lt_table.iterrows():
            if not pd.isna(lt_table.at[index, 'latitude']) and not pd.isna(lt_table.at[index, 'longitude']):
                 lt_table.at[index, column_for_data] = get_soil_temp_at_closest_loc(min_table, column_for_data, lt_table.at[index, 'latitude'], lt_table.at[index, 'longitude'])
    pass

In [ ]:
# Get some temp for NC-2010-003 and PA-2010-003 (in order)
#yearly_minimum_30_year_cal_NA = pd.read_csv('yearly_minimum/NA/yearly_minimum_30_year_compile_NA_with_cal.csv')
#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_50th_percentile', 35.59, -82.57)
#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_25th_percentile', 35.59, -82.57)
#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_absolute_min', 35.59, -82.57)

#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_50th_percentile', 39.96, -72.39)
#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_25th_percentile', 39.96, -72.39)
#get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_NA, 'recent_30_year_absolute_min', 39.96, -72.39)

In [ ]:
yearly_minimum_30_year_cal_EA = pd.read_csv('yearly_minimum/EA/yearly_minimum_30_year_compile_EA_with_cal.csv')
get_soil_temp_at_closest_loc(yearly_minimum_30_year_cal_EA, 'recent_30_year_averaged_min', 30, 103)

In [ ]:
#ltmultipred = pd.read_excel("C:/Users/xuyin/Box/1-Sacks Lab Box Folder/Xuying Zheng' Folder/Freeze Test Substudy/2022-2023 Freeze Test/ltmulti2_pred_2y_avg_cleaned.xlsx", 
 #                           sheet_name='ltmulti2_pred_2y_avg_cleaned')
ltmultipred = pd.read_excel("ltmulti2_pred_2y_avg_cleaned.xlsx", sheet_name='ltmulti2_pred_2y_avg_cleaned')
retrieve_soil_temp_for_entry_table(ltmultipred, yearly_720hmean_minimum_30_year_cal_EA, 
                                   ['recent_10_year_averaged_min', 'recent_20_year_averaged_min', 'recent_30_year_averaged_min',
                                   'recent_10_year_50th_percentile', 'recent_20_year_50th_percentile', 'recent_30_year_50th_percentile'])
print(ltmultipred)
ltmultipred.to_csv("ltmultipred_2yr_clean_w_720hmean_soiltemp.csv", index = False)

In [ ]:
# test
plt.figure(figsize=(10, 6))
plt.scatter(loc_lons_masked, loc_lats_masked, c=loc_hourly_mins_celsius_masked, cmap='viridis', s=6, marker='o', edgecolors='none')
plt.colorbar(label='Temperature in Celsius (°C)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Soil Annual Minimum Temperature at 10 cm Depth')
plt.grid(True)
plt.show()

In [ ]:
print(grb1.validDate)
print(grb2.validDate)
print(type(grb1.validDate))

In [ ]:
print(type(grb1.values))
print(grb1.values.shape)
print(grb2.values.shape)

In [ ]:
lats, lons = grb1.latlons()
print(len(lats.flatten())); print(len(lons.flatten())); print(type(lats)); print(type(lons))

In [ ]:
vals = grb1.values.flatten()
print(vals.dtype)
plt.hist(vals,bins = 30)

In [ ]:
for grb in grbs[1:2]:
    print(grb.values.flatten)
    print(grb.parameterName)
    print(grb.latlons())